In [ ]:
import os
import json
import csv
from pathlib import Path
from collections import defaultdict
from semanticher.data import load_label_class

In [ ]:
# =========================
# Base folders
# =========================
REPO_ROOT       = Path().resolve().parents[1]
BASE_RESULT_DIR = REPO_ROOT / "results" / "main_msv_base"
CANDIDATE_DIR   = BASE_RESULT_DIR / "Result_msv_base" / "Candidate_msv_base"
RESULT_DIR      = BASE_RESULT_DIR / "Result_msv_base" / "msv_base"
EVAL_DIR        = BASE_RESULT_DIR / "Eval_msv_base"   / "msv_base"

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR,   exist_ok=True)

df_labels = load_label_class()

# URI -> canonical name mapping
MAPPING_FILE = REPO_ROOT / "data" / "ontology" / "name_uri_mapping.json"
with open(MAPPING_FILE, "r", encoding="utf-8") as f:
    mapping = json.load(f)

uri_to_name = {}
for name, uris in mapping["classes"].items():
    for uri in uris:
        uri_to_name[uri] = name


# =========================
# Model aliases / experiments
# =========================
MODEL_ALIAS = {
    "deepseek-chat":    "dsk",
    "gemini-2.0-flash": "gem",
    "gpt-4.1-mini":     "gpt",
}

EXPERIMENTS = [
    ["deepseek-chat"],
    ["gemini-2.0-flash"],
    ["gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash"],
    ["deepseek-chat", "gpt-4.1-mini"],
    ["gemini-2.0-flash", "gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash", "gpt-4.1-mini"],
]

EVAL_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


def build_experiment_tag(model_names):
    aliases = [MODEL_ALIAS[m] for m in model_names]
    prefix  = {1: "single", 2: "two", 3: "three"}[len(model_names)]
    return f"{prefix}_{'_'.join(aliases)}"


def build_candidate_path(model_names):
    tag = build_experiment_tag(model_names)
    return CANDIDATE_DIR / f"candidate_msv_base_{tag}.json"


def build_result_dir(model_names):
    tag     = build_experiment_tag(model_names)
    out_dir = RESULT_DIR / f"msv_base_{tag}"
    os.makedirs(out_dir, exist_ok=True)
    return out_dir


def build_eval_dir(model_names):
    tag     = build_experiment_tag(model_names)
    out_dir = EVAL_DIR / f"msv_base_{tag}"
    os.makedirs(out_dir, exist_ok=True)
    return out_dir

In [ ]:
# =========================
# Helper functions
# =========================
def clean_label(x):
    if x is None:
        return ""
    s = str(x).strip()
    return "" if s in ("", "-", "nan", "NaN") else s


def flatten_list_to_set(paths):
    nodes = set()
    for path in paths:
        if isinstance(path, list):
            for item in path:
                if item is not None and str(item).strip():
                    nodes.add(str(item).strip())
    return nodes


def path_to_list(path_item):
    """
    Candidate path format:
      ["Level1_URI", conf]
      ["Level1_URI", "Level2_URI", conf]
    Convert URI to name, return:
      [Level1_name] / [Level1_name, Level2_name], conf
    """
    if not isinstance(path_item, (list, tuple)):
        return None, None

    if len(path_item) == 2:
        level1, conf = path_item
        level1 = uri_to_name.get(level1)
        if not level1:
            return None, None
        level1 = clean_label(level1)
        if not level1:
            return None, None
        try:
            conf = float(conf)
        except Exception:
            return None, None
        return [level1], conf

    if len(path_item) == 3:
        level1, level2, conf = path_item
        level1 = uri_to_name.get(level1)
        level2 = uri_to_name.get(level2)
        if not level1 or not level2:
            return None, None
        level1 = clean_label(level1)
        level2 = clean_label(level2)
        if not level1 or not level2:
            return None, None
        try:
            conf = float(conf)
        except Exception:
            return None, None
        if level1 == level2:
            return [level1], conf
        return [level1, level2], conf

    return None, None


def prf_only(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall    = tp / (tp + fn) if tp + fn else 0.0
    f1        = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

In [ ]:
# =========================
# Ground truth & evaluation
# =========================
def build_gt_paths(label_row):
    gt_paths = []

    for i in [1, 2, 3]:
        l1 = clean_label(label_row.get(f"class{i}_level1"))
        l2 = clean_label(label_row.get(f"class{i}_level2"))

        if l1 and l2:
            gt_paths.append([l1, l2])
        elif l1:
            gt_paths.append([l1])

    return gt_paths


def prepare_eval_data(result_pred, df_labels):
    eval_json = []

    for r in result_pred:
        table_id  = r["table_id"]
        column_id = r["column_id"]

        label_row = df_labels[
            (df_labels["table_id"].astype(str)  == str(table_id)) &
            (df_labels["column_id"].astype(int) == int(column_id))
        ].iloc[0]

        eval_json.append({
            "table_id":    r["table_id"],
            "table_name":  r["table_name"],
            "column_id":   r["column_id"],
            "column_name": r["column_name"],
            "pred_paths":  r.get("paths", []) or [],
            "gt_paths":    build_gt_paths(label_row),
        })

    return eval_json


def path_level_f1_precision_recall(eval_json):
    scores = []

    for row in eval_json:
        pred = {tuple(p) for p in row["pred_paths"]}
        gt   = {tuple(p) for p in row["gt_paths"]}

        tp = len(pred & gt)
        fp = len(pred - gt)
        fn = len(gt - pred)

        precision, recall, f1 = prf_only(tp, fp, fn)
        scores.append({"tp": tp, "fp": fp, "fn": fn,
                        "precision": precision, "recall": recall, "f1": f1})

    macro_precision = sum(s["precision"] for s in scores) / len(scores) if scores else 0.0
    macro_recall    = sum(s["recall"]    for s in scores) / len(scores) if scores else 0.0
    macro_f1        = sum(s["f1"]        for s in scores) / len(scores) if scores else 0.0

    tp_total = sum(s["tp"] for s in scores)
    fp_total = sum(s["fp"] for s in scores)
    fn_total = sum(s["fn"] for s in scores)

    micro_precision, micro_recall, micro_f1 = prf_only(tp_total, fp_total, fn_total)

    return {
        "macro": {"precision": macro_precision, "recall": macro_recall, "f1": macro_f1},
        "micro": {"precision": micro_precision, "recall": micro_recall, "f1": micro_f1},
    }


def node_level_f1_precision_recall(eval_json):
    scores = []

    for row in eval_json:
        pred_nodes = flatten_list_to_set(row["pred_paths"])
        gt_nodes   = flatten_list_to_set(row["gt_paths"])

        tp = len(pred_nodes & gt_nodes)
        fp = len(pred_nodes - gt_nodes)
        fn = len(gt_nodes - pred_nodes)

        precision, recall, f1 = prf_only(tp, fp, fn)
        scores.append({"tp": tp, "fp": fp, "fn": fn,
                        "precision": precision, "recall": recall, "f1": f1})

    macro_precision = sum(s["precision"] for s in scores) / len(scores) if scores else 0.0
    macro_recall    = sum(s["recall"]    for s in scores) / len(scores) if scores else 0.0
    macro_f1        = sum(s["f1"]        for s in scores) / len(scores) if scores else 0.0

    tp_total = sum(s["tp"] for s in scores)
    fp_total = sum(s["fp"] for s in scores)
    fn_total = sum(s["fn"] for s in scores)

    micro_precision, micro_recall, micro_f1 = prf_only(tp_total, fp_total, fn_total)

    return {
        "macro": {"precision": macro_precision, "recall": macro_recall, "f1": macro_f1},
        "micro": {"precision": micro_precision, "recall": micro_recall, "f1": micro_f1},
    }


def evaluate_result(eval_json):
    return {
        "path": path_level_f1_precision_recall(eval_json),
        "node": node_level_f1_precision_recall(eval_json),
    }


def save_eval_txt(eval_result, out_path):
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("=== PATH LEVEL ===\n")
        f.write("=== MACRO ===\n")
        f.write(json.dumps(eval_result["path"]["macro"], ensure_ascii=False, indent=2))
        f.write("\n")
        f.write("=== MICRO ===\n")
        f.write(json.dumps(eval_result["path"]["micro"], ensure_ascii=False, indent=2))
        f.write("\n\n")
        f.write("=== NODE LEVEL ===\n")
        f.write("=== MACRO ===\n")
        f.write(json.dumps(eval_result["node"]["macro"], ensure_ascii=False, indent=2))
        f.write("\n")
        f.write("=== MICRO ===\n")
        f.write(json.dumps(eval_result["node"]["micro"], ensure_ascii=False, indent=2))
        f.write("\n")

In [ ]:
# =========================
# Soft voting (post mode)
# =========================
def group_candidate_rows(candidate_rows):
    grouped = defaultdict(lambda: {
        "table_id":    None,
        "table_name":  None,
        "column_name": None,
        "column_id":   None,
        "by_model":    {}
    })

    for row in candidate_rows:
        key = (row["table_id"], row["column_id"])
        grouped[key]["table_id"]    = row["table_id"]
        grouped[key]["table_name"]  = row["table_name"]
        grouped[key]["column_name"] = row["column_name"]
        grouped[key]["column_id"]   = row["column_id"]
        grouped[key]["by_model"][row["model"]] = row.get("class", []) or []

    return grouped


def soft_vote(grouped_rows, model_names):
    """
    Post mode: all candidate paths enter soft voting directly.
    URI → name conversion happens inside path_to_list.
    score(path) = sum(conf_i) / N
    """
    result     = []
    num_models = len(model_names)

    for _, row in grouped_rows.items():
        score_map = defaultdict(float)

        for model_name in model_names:
            path_items = row["by_model"].get(model_name, [])

            for item in path_items:
                path_list, conf = path_to_list(item)
                if path_list is None or conf is None:
                    continue
                score_map[tuple(path_list)] += conf / num_models

        ranked_paths = sorted(score_map.items(), key=lambda x: x[1], reverse=True)

        result.append({
            "table_id":    row["table_id"],
            "table_name":  row["table_name"],
            "column_name": row["column_name"],
            "column_id":   row["column_id"],
            "paths":       [list(p) for p, _ in ranked_paths],
            "scores":      [round(s, 6) for _, s in ranked_paths],
        })

    return result


def apply_threshold(voted_result, threshold):
    filtered = []

    for row in voted_result:
        new_paths  = []
        new_scores = []

        for path, score in zip(row.get("paths", []), row.get("scores", [])):
            if score >= threshold:
                new_paths.append(path)
                new_scores.append(score)

        filtered.append({
            "table_id":    row["table_id"],
            "table_name":  row["table_name"],
            "column_name": row["column_name"],
            "column_id":   row["column_id"],
            "paths":       new_paths,
            "scores":      new_scores,
        })

    return filtered

In [ ]:
# =========================
# Single experiment runner
# =========================
def run_single_experiment(model_names):
    tag            = build_experiment_tag(model_names)
    candidate_file = build_candidate_path(model_names)
    result_dir     = build_result_dir(model_names)
    eval_dir       = build_eval_dir(model_names)

    print("=" * 80)
    print(f"Running experiment: {tag}")
    print("Models:    ", model_names)
    print("Candidate: ", candidate_file)
    print("=" * 80)

    with open(candidate_file, encoding="utf-8") as f:
        candidate_rows = json.load(f)

    grouped_rows = group_candidate_rows(candidate_rows)
    voted_result = soft_vote(grouped_rows, model_names)

    # save raw voted result (before thresholding)
    raw_vote_path = result_dir / f"msv_base_{tag}_raw_vote.json"
    with open(raw_vote_path, "w", encoding="utf-8") as f:
        json.dump(voted_result, f, ensure_ascii=False, indent=2)

    scores       = {}
    full_results = {}

    print(f"\n===== threshold sweep: {tag} =====")

    for threshold in EVAL_THRESHOLDS:
        result_pred = apply_threshold(voted_result, threshold)

        ttag        = str(threshold).replace(".", "")
        result_path = result_dir / f"msv_base_{tag}_t{ttag}.json"
        with open(result_path, "w", encoding="utf-8") as f:
            json.dump(result_pred, f, ensure_ascii=False, indent=2)

        eval_json   = prepare_eval_data(result_pred, df_labels)
        eval_result = evaluate_result(eval_json)

        f1                      = eval_result["node"]["micro"]["f1"]
        scores[threshold]       = f1
        full_results[threshold] = eval_result

        print(f"  threshold={threshold:.1f}  node_micro_f1={f1:.6f}")

    best_t = max(scores, key=scores.get)
    print(f"\n  best threshold: {best_t}  f1={scores[best_t]:.6f}")

    # save eval txt files
    best_dir = eval_dir / "best_eval"
    os.makedirs(best_dir, exist_ok=True)

    for threshold in EVAL_THRESHOLDS:
        ttag     = str(threshold).replace(".", "")
        t_dir    = eval_dir / f"t{ttag}_eval"
        os.makedirs(t_dir, exist_ok=True)
        out_file = t_dir / f"msv_base_{tag}_t{ttag}_eval.txt"
        save_eval_txt(full_results[threshold], out_file)

    best_out = best_dir / f"msv_base_{tag}_best_t{str(best_t).replace('.', '')}_eval.txt"
    save_eval_txt(full_results[best_t], best_out)

    print(f"  Saved eval -> {eval_dir}")

    return scores, full_results, best_t

In [ ]:
# =========================
# Run all & summary
# =========================
def run_all_experiments():
    all_scores       = {}
    all_full_results = {}
    all_best_t       = {}

    for model_names in EXPERIMENTS:
        tag = build_experiment_tag(model_names)
        scores, full_results, best_t = run_single_experiment(model_names)
        all_scores[tag]       = scores
        all_full_results[tag] = full_results
        all_best_t[tag]       = best_t

    # summary csv
    summary_csv = EVAL_DIR / "threshold_summary.csv"
    with open(summary_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["experiment"] + [f"t{str(t).replace('.','')}" for t in EVAL_THRESHOLDS] + ["best"])

        for tag in all_scores:
            row = [tag]
            for t in EVAL_THRESHOLDS:
                row.append(round(all_scores[tag][t], 6) if t in all_scores[tag] else "")
            row.append(all_best_t[tag])
            writer.writerow(row)

    # summary txt
    summary_txt = EVAL_DIR / "summary.txt"
    with open(summary_txt, "w", encoding="utf-8") as f:
        f.write("MSV Base threshold sweep\n")
        f.write("Best threshold criterion: NODE / MICRO / F1\n\n")

        for tag in all_scores:
            f.write(f"{tag}\n")
            for t in EVAL_THRESHOLDS:
                f.write(f"  {t:.1f} -> {all_scores[tag][t]:.6f}\n")
            f.write(f"  best = {all_best_t[tag]}\n\n")

    print(f"Saved summary -> {summary_csv}")
    print(f"Saved summary -> {summary_txt}")


run_all_experiments()